## 16. تشخیص Duplicate

آگهی‌های تکراری می‌توانند حجم عرضه را بیش از واقع نشان دهند.

تیم باید حداقل دو سطح Duplicate را بررسی کند:

1. **Exact Duplicate:** رکوردهای کاملاً یکسان
2. **Probable Duplicate:** آگهی‌های احتمالاً مربوط به یک ملک

ویژگی‌های احتمالی برای Duplicate تقریبی:

- شهر و محله
- مختصات نزدیک
- مساحت
- تعداد اتاق
- طبقه
- قیمت مشابه
- متن مشابه
- ماه ثبت
- نوع کاربر

حذف Duplicate احتمالی باید محافظه‌کارانه و قابل Audit باشد. در صورت عدم حذف، اثر آن بر
شاخص عرضه باید در تحلیل حساسیت بررسی شود.

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist
from itertools import combinations
import hashlib

from datasets import load_dataset

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset = load_dataset("divarofficial/real_estate_ads")

# df = pd.read_feather("../Outputs/02_df.feather")

In [ ]:
# 9.1 Define Candidate Duplicate Keys

candidate_key_cols = [
    "city_slug",
    "neighborhood_slug",
    "cat3_slug",
    "building_size",
    "rooms_count_numeric",
    "construction_year_numeric",
]

candidate_duplicate_mask = df.duplicated(
    subset=candidate_key_cols,
    keep=False
)

print(
    "Rows sharing the candidate duplicate key:",
    candidate_duplicate_mask.sum()
)

print(
    "Candidate duplicate row rate (%):",
    round(candidate_duplicate_mask.mean() * 100, 2)
)

candidate_group_sizes = (
    df.loc[candidate_duplicate_mask]
      .groupby(candidate_key_cols, dropna=False)
      .size()
      .sort_values(ascending=False)
)

print("\nNumber of candidate groups:", len(candidate_group_sizes))

print("\nLargest candidate groups:")
print(candidate_group_sizes.head(20).to_string())

In [ ]:
# 9.2 High-Confidence Content Duplicate Screening

content_key_cols = [
    "city_slug",
    "cat3_slug",
    "title_clean",
    "description_clean",
    "building_size",
]

# Create a compact fingerprint instead of grouping long text directly
df["content_fingerprint"] = pd.util.hash_pandas_object(
    df[content_key_cols],
    index=False
)

content_duplicate_mask = df.duplicated(
    subset=["content_fingerprint"],
    keep=False
)

print(
    "Rows sharing the same content fingerprint:",
    content_duplicate_mask.sum()
)

print(
    "Content duplicate row rate (%):",
    round(content_duplicate_mask.mean() * 100, 4)
)

content_group_sizes = (
    df.loc[content_duplicate_mask, "content_fingerprint"]
      .value_counts()
)

print(
    "\nNumber of repeated content groups:",
    len(content_group_sizes)
)

print("\nLargest repeated content groups:")
print(content_group_sizes.head(20).to_string())

In [ ]:
# 9.3 Strengthen Duplicate Evidence with Price and Time

duplicate_candidates = df.loc[
    content_duplicate_mask,
    [
        "content_fingerprint",
        "created_at_month",
        "cat2_slug",
        "cat3_slug",
        "city_slug",
        "neighborhood_slug",
        "building_size",
        "price_value",
        "rent_value",
        "credit_value",
        "title_clean",
    ]
].copy()

duplicate_candidates = duplicate_candidates.sort_values(
    ["content_fingerprint", "created_at_month"]
)

candidate_group_summary = (
    duplicate_candidates
    .groupby("content_fingerprint")
    .agg(
        row_count=("content_fingerprint", "size"),
        month_count=("created_at_month", "nunique"),
        price_count=("price_value", "nunique"),
        rent_count=("rent_value", "nunique"),
        credit_count=("credit_value", "nunique"),
    )
)

print("Repeated content groups:", len(candidate_group_summary))

print(
    "\nGroups with same month:",
    (candidate_group_summary["month_count"] == 1).sum()
)

print(
    "Groups with one unique sale price:",
    (candidate_group_summary["price_count"] <= 1).sum()
)

print(
    "Groups with one unique rent value:",
    (candidate_group_summary["rent_count"] <= 1).sum()
)

print(
    "Groups with one unique credit value:",
    (candidate_group_summary["credit_count"] <= 1).sum()
)

print("\nSample group summary:")
print(
    candidate_group_summary
    .sort_values("row_count", ascending=False)
    .head(20)
    .to_string()
)

In [ ]:
# 9.4 Transaction-Aware Duplicate Consistency

group_base = df.loc[
    content_duplicate_mask,
    [
        "content_fingerprint",
        "cat2_slug",
        "created_at_month",
        "price_value",
        "price_mode",
        "rent_value",
        "rent_mode",
        "credit_value",
        "credit_mode",
    ]
].copy()

sale_group_data = group_base[
    group_base["cat2_slug"].isin([
        "residential-sell",
        "commercial-sell"
    ])
]

rent_group_data = group_base[
    group_base["cat2_slug"].isin([
        "residential-rent",
        "commercial-rent"
    ])
]

sale_group_summary = (
    sale_group_data
    .groupby("content_fingerprint")
    .agg(
        row_count=("content_fingerprint", "size"),
        month_count=("created_at_month", "nunique"),
        price_value_count=("price_value", "nunique"),
        price_mode_count=("price_mode", "nunique"),
    )
)

rent_group_summary = (
    rent_group_data
    .groupby("content_fingerprint")
    .agg(
        row_count=("content_fingerprint", "size"),
        month_count=("created_at_month", "nunique"),
        rent_value_count=("rent_value", "nunique"),
        credit_value_count=("credit_value", "nunique"),
        rent_mode_count=("rent_mode", "nunique"),
        credit_mode_count=("credit_mode", "nunique"),
    )
)

print("--- SALE CONTENT-DUPLICATE GROUPS ---")
print("Groups:", len(sale_group_summary))
print(
    "Same month:",
    (sale_group_summary["month_count"] == 1).sum()
)
print(
    "Same non-missing price:",
    (sale_group_summary["price_value_count"] == 1).sum()
)

print("\n--- RENT CONTENT-DUPLICATE GROUPS ---")
print("Groups:", len(rent_group_summary))
print(
    "Same month:",
    (rent_group_summary["month_count"] == 1).sum()
)
print(
    "Same non-missing rent:",
    (rent_group_summary["rent_value_count"] == 1).sum()
)
print(
    "Same non-missing credit:",
    (rent_group_summary["credit_value_count"] == 1).sum()
)

In [ ]:
# 9.5 Identify High-Confidence Duplicate Groups

sale_high_confidence = (
    (sale_group_summary["row_count"] >= 2)
    & (sale_group_summary["month_count"] == 1)
    & (sale_group_summary["price_value_count"] == 1)
    & (sale_group_summary["price_mode_count"] <= 1)
)

rent_high_confidence = (
    (rent_group_summary["row_count"] >= 2)
    & (rent_group_summary["month_count"] == 1)
    & (rent_group_summary["rent_value_count"] == 1)
    & (rent_group_summary["credit_value_count"] == 1)
    & (rent_group_summary["rent_mode_count"] <= 1)
    & (rent_group_summary["credit_mode_count"] <= 1)
)

sale_high_conf_groups = sale_group_summary.loc[sale_high_confidence]
rent_high_conf_groups = rent_group_summary.loc[rent_high_confidence]

print("--- HIGH-CONFIDENCE SALE DUPLICATES ---")
print("Groups:", len(sale_high_conf_groups))
print("Rows involved:", sale_high_conf_groups["row_count"].sum())

print("\n--- HIGH-CONFIDENCE RENT DUPLICATES ---")
print("Groups:", len(rent_high_conf_groups))
print("Rows involved:", rent_high_conf_groups["row_count"].sum())

In [ ]:
# 9.6 Flag High-Confidence Duplicate Candidates

sale_high_conf_fps = sale_high_conf_groups.index
rent_high_conf_fps = rent_high_conf_groups.index

df["high_confidence_duplicate_candidate"] = False

df.loc[
    df["content_fingerprint"].isin(sale_high_conf_fps)
    & df["cat2_slug"].isin(["residential-sell", "commercial-sell"]),
    "high_confidence_duplicate_candidate"
] = True

df.loc[
    df["content_fingerprint"].isin(rent_high_conf_fps)
    & df["cat2_slug"].isin(["residential-rent", "commercial-rent"]),
    "high_confidence_duplicate_candidate"
] = True

print(
    "High-confidence duplicate candidate rows:",
    df["high_confidence_duplicate_candidate"].sum()
)

print(
    "Rate (%):",
    round(
        df["high_confidence_duplicate_candidate"].mean() * 100,
        4
    )
)